# Fine-Tuning de YOLOv8 — Billetes Mexicanos (TT2026-2_IA16)

Notebook reproducible de entrenamiento y evaluacion. Entrenamiento con descongelamiento gradual en 3 fases, semilla fija. Pensado para Kaggle / Google Colab con GPU (T4 / P100).

**Estado:** base. La ejecucion de las corridas y la evaluacion se completan con el dataset etiquetado.

## 1. Instalacion de dependencias

In [1]:
!pip install ultralytics roboflow imagehash -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.2/97.2 kB 10.9 MB/s eta 0:00:00


## 2. Imports y semilla fija (reproducibilidad)

In [2]:
import os, random, shutil, yaml
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, torch
from PIL import Image
import imagehash
from ultralytics import YOLO

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

print('GPU disponible:', torch.cuda.is_available())
print('Dispositivo:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
GPU disponible: True
Dispositivo: Tesla T4


## 3. Dataset

Descarga del dataset exportado de Roboflow (formato YOLOv8). Sustituye los datos por los de tu workspace/proyecto/version. El archivo `data.yaml` define las 6 clases y las rutas train/val/test.

In [3]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="HcBhrngFgtCWTuxeb31A")
project = rf.workspace("luis-ervey-guzman-hernandez").project("billetes-mexicanos-b3s1u")
version = project.version(1)
dataset = version.download("yolov8")
print('Descargado en:', dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Billetes-Mexicanos-1 in yolov8:: 100%|██████████| 5205/5205 [00:00<00:00, 6046.44it/s]


Descargado en: /content/Billetes-Mexicanos-1


## 4.Inventario, verificación de bbox y hashes


In [4]:
root  = Path(dataset.location)
names = yaml.safe_load(open(root/'data.yaml'))['names']
K = len(names)

items, no_bbox, vacias = [], 0, 0      # items: (img, lbl, Counter de clases)
for split in ['train','valid','test']:
    d = root/split/'images'
    if not d.exists(): continue
    for img in sorted(d.glob('*')):
        lbl = root/split/'labels'/(img.stem + '.txt')
        if not lbl.exists(): continue
        filas = [l.split() for l in lbl.read_text().splitlines() if l.strip()]
        if not filas: vacias += 1
        no_bbox += sum(1 for f in filas if len(f) != 5)
        items.append((img, lbl, Counter(int(f[0]) for f in filas)))

n = len(items)
ann_tot = Counter()
for _,_,c in items: ann_tot.update(c)

print(f"Imágenes en el pool : {n}")
print(f"Sin etiqueta        : {vacias}     <- debe ser 0")
print(f"Etiquetas NO bbox   : {no_bbox}     <- debe ser 0\n")
print(f"{'Clase':<16}{'Anotaciones':>12}")
for i,nm in enumerate(names): print(f"{nm:<16}{ann_tot[i]:>12}")
print(f"{'TOTAL':<16}{sum(ann_tot.values()):>12}")

print("\nCalculando hashes perceptuales...")
hs = [int(str(imagehash.phash(Image.open(p))), 16) for p,_,_ in items]
print("Listo.")

Imágenes en el pool : 2600
Sin etiqueta        : 0     <- debe ser 0
Etiquetas NO bbox   : 3031     <- debe ser 0

Clase            Anotaciones
billete_100              579
billete_1000             404
billete_20               514
billete_200              407
billete_50               438
billete_500              717
TOTAL                   3059

Calculando hashes perceptuales...
Listo.


## 4.Inventario, verificación de bbox y hashes


In [5]:
# Convertir polígonos a bounding box (rectángulo mínimo que los encierra)
conv, ya_bbox, degenerados = 0, 0, 0
items_fix = []

for (img, lbl, _) in items:
    filas = [l.split() for l in lbl.read_text().splitlines() if l.strip()]
    nuevas = []
    for f in filas:
        cls = int(f[0])
        v = [float(x) for x in f[1:]]
        if len(v) == 4:                      # ya era bbox
            cx, cy, w, h = v
            x0, x1 = cx - w/2, cx + w/2
            y0, y1 = cy - h/2, cy + h/2
            ya_bbox += 1
        else:                                # polígono -> caja envolvente
            xs, ys = v[0::2], v[1::2]
            x0, x1 = min(xs), max(xs)
            y0, y1 = min(ys), max(ys)
            conv += 1
        # recortar al marco de la imagen
        x0, y0 = max(0.0, x0), max(0.0, y0)
        x1, y1 = min(1.0, x1), min(1.0, y1)
        w, h = x1 - x0, y1 - y0
        if w <= 0 or h <= 0:
            degenerados += 1
            continue
        nuevas.append(f"{cls} {(x0+x1)/2:.6f} {(y0+y1)/2:.6f} {w:.6f} {h:.6f}")
    lbl.write_text("\n".join(nuevas) + ("\n" if nuevas else ""))
    items_fix.append((img, lbl, Counter(int(x.split()[0]) for x in nuevas)))

items = items_fix

# ---------- RE-VERIFICACIÓN ----------
no_bbox2, ann2, vac2 = 0, Counter(), 0
for _, lbl, c in items:
    filas = [l.split() for l in lbl.read_text().splitlines() if l.strip()]
    if not filas: vac2 += 1
    no_bbox2 += sum(1 for f in filas if len(f) != 5)
    ann2.update(c)

print(f"Polígonos convertidos a bbox : {conv}")
print(f"Ya venían como bbox          : {ya_bbox}")
print(f"Descartadas (área cero)      : {degenerados}\n")
print(f"Etiquetas NO bbox ahora      : {no_bbox2}   <- debe ser 0")
print(f"Imágenes sin etiqueta        : {vac2}   <- debe ser 0\n")
print(f"{'Clase':<16}{'Anotaciones':>12}")
for i, nm in enumerate(names): print(f"{nm:<16}{ann2[i]:>12}")
print(f"{'TOTAL':<16}{sum(ann2.values()):>12}")

Polígonos convertidos a bbox : 3031
Ya venían como bbox          : 28
Descartadas (área cero)      : 0

Etiquetas NO bbox ahora      : 0   <- debe ser 0
Imágenes sin etiqueta        : 0   <- debe ser 0

Clase            Anotaciones
billete_100              579
billete_1000             404
billete_20               514
billete_200              407
billete_50               438
billete_500              717
TOTAL                   3059


## 5. Agrupar escenas (repetible: ajusta UMBRAL y vuelve a correr SOLO esta)



In [13]:
UMBRAL = 8        # <<< el parámetro a calibrar

try: pc = int.bit_count
except AttributeError: pc = lambda x: bin(x).count('1')

parent = list(range(n))
def find(x):
    while parent[x]!=x: parent[x]=parent[parent[x]]; x=parent[x]
    return x
def union(a,b):
    ra,rb = find(a),find(b)
    if ra!=rb: parent[rb]=ra

for i in range(n):
    hi = hs[i]
    for j in range(i+1, n):
        if pc(hi ^ hs[j]) <= UMBRAL: union(i,j)

gd = defaultdict(list)
for i in range(n): gd[find(i)].append(i)
grupos = list(gd.values())

tam = Counter(len(g) for g in grupos)
print(f"UMBRAL = {UMBRAL}")
print(f"Grupos formados : {len(grupos)}   (de {n} imágenes)")
print(f"Promedio        : {n/len(grupos):.2f} imágenes por grupo")
print(f"Grupo más grande: {max(len(g) for g in grupos)} imágenes\n")
print("Distribución de tamaños:")
for k in sorted(tam): print(f"   {k:>3} imagen(es): {tam[k]:>5} grupos")

def corto(p):
    s = p.name
    return s.split('.rf.')[0] if '.rf.' in s else s

print("\n--- 8 GRUPOS MÁS GRANDES ---")
for g in sorted(grupos, key=len, reverse=True)[:8]:
    print(f"[{len(g):>2}] " + ", ".join(corto(items[i][0]) for i in g[:10]) + (" ..." if len(g)>10 else ""))

random.seed(SEED)
med = [g for g in grupos if 2 <= len(g) <= 5]
print(f"\n--- 10 GRUPOS MEDIANOS al azar (de {len(med)}) ---")
for g in random.sample(med, min(10,len(med))):
    print(f"[{len(g)}] " + ", ".join(corto(items[i][0]) for i in g))

UMBRAL = 8
Grupos formados : 2416   (de 2600 imágenes)
Promedio        : 1.08 imágenes por grupo
Grupo más grande: 27 imágenes

Distribución de tamaños:
     1 imagen(es):  2333 grupos
     2 imagen(es):    65 grupos
     3 imagen(es):     7 grupos
     4 imagen(es):     3 grupos
     6 imagen(es):     2 grupos
     9 imagen(es):     1 grupos
    12 imagen(es):     2 grupos
    14 imagen(es):     1 grupos
    18 imagen(es):     1 grupos
    27 imagen(es):     1 grupos

--- 8 GRUPOS MÁS GRANDES ---
[27] IMG_5872_jpeg, IMG_6021_jpeg, IMG_6059_jpeg, IMG_6064_jpeg, IMG_6070_jpeg, IMG_6117_jpeg, IMG_6475_JPG, IMG_6479_JPG, IMG_6481_JPG, IMG_6655_JPG ...
[18] IMG_3071_jpeg, IMG_3072_jpeg, IMG_3073_jpeg, IMG_3075_jpeg, IMG_3076_jpeg, IMG_3077_jpeg, IMG_3078_jpeg, IMG_3082_jpeg, IMG_3083_jpeg, IMG_3085_jpeg ...
[14] IMG_5809_jpeg, IMG_5812_jpeg, IMG_5814_jpeg, IMG_5829_jpeg, IMG_6001_jpeg, IMG_6473_JPG, IMG_6482_JPG, IMG_6558_JPG, IMG_6711_JPG, IMG_7549_JPG ...
[12] IMG_5790_jpeg, IMG_5816_jpe

In [15]:
#5-BIS — prueba varios umbrales de un jalón
import re
try: pc = int.bit_count
except AttributeError: pc = lambda x: bin(x).count('1')

# distancias <= 20, una sola vez
pares = []
for i in range(n):
    hi = hs[i]
    for j in range(i+1, n):
        d = pc(hi ^ hs[j])
        if d <= 20: pares.append((d, i, j))
pares.sort()
print(f"Pares con distancia <= 20: {len(pares)}\n")

def num(p):
    m = re.search(r'(\d{3,})', p.name)
    return int(m.group(1)) if m else -1
nums = [num(p) for p,_,_ in items]

print(f"{'UMB':>4}{'Grupos':>8}{'Solos':>8}{'Mayor':>7}{'ImgsEnGrupo':>13}{'%coherentes':>13}")
for T in [6, 8, 10, 12, 14, 16]:
    par = list(range(n))
    def find(x):
        while par[x]!=x: par[x]=par[par[x]]; x=par[x]
        return x
    for d,i,j in pares:
        if d > T: break
        a,b = find(i), find(j)
        if a!=b: par[b]=a
    gd = defaultdict(list)
    for i in range(n): gd[find(i)].append(i)
    gs = list(gd.values())
    multi = [g for g in gs if len(g) > 1]
    coher = sum(1 for g in multi if max(nums[i] for i in g) - min(nums[i] for i in g) <= 10)
    solos = sum(1 for g in gs if len(g) == 1)
    print(f"{T:>4}{len(gs):>8}{solos:>8}{max(len(g) for g in gs):>7}"
          f"{n-solos:>13}{100*coher/max(len(multi),1):>12.0f}%")

Pares con distancia <= 20: 62092

 UMB  Grupos   Solos  Mayor  ImgsEnGrupo  %coherentes
   6    2513    2469     13          131          66%
   8    2416    2333     27          267          58%
  10    2236    2090    129          510          55%
  12    1851    1626    319          974          44%
  14    1001     836   1318         1764          26%
  16     175     155   2401         2445          45%


## 6. Split estratificado por grupo (con la corrección)


In [17]:
OUT = Path('/content/dataset_scene')
if OUT.exists(): shutil.rmtree(OUT)
RATIO = {'train':0.70, 'valid':0.15, 'test':0.15}

g_cnt = [[sum(items[i][2][k] for i in g) for k in range(K)] for g in grupos]
g_img = [len(g) for g in grupos]
tot_c = [sum(gc[k] for gc in g_cnt) for k in range(K)]

meta_c = {s:[t*r for t in tot_c] for s,r in RATIO.items()}
meta_i = {s:n*r for s,r in RATIO.items()}
ac_c   = {s:[0]*K for s in RATIO}
ac_i   = {s:0 for s in RATIO}
asig, colocado = {}, 0

for gi in sorted(range(len(grupos)), key=lambda i: -sum(g_cnt[i])):
    colocado += g_img[gi]
    prog = colocado / n                      # avance global esperado
    mejor, cmin = None, None
    for s in RATIO:
        ac_i[s] += g_img[gi]                 # colocar tentativamente
        for k in range(K): ac_c[s][k] += g_cnt[gi][k]
        # costo = qué tan desparejos quedan los 3 conjuntos, en imágenes y en cada clase
        c  = sum((ac_i[t]/meta_i[t] - prog)**2 for t in RATIO)
        c += sum((ac_c[t][k]/meta_c[t][k] - prog)**2
                 for t in RATIO for k in range(K) if tot_c[k])
        ac_i[s] -= g_img[gi]                 # deshacer
        for k in range(K): ac_c[s][k] -= g_cnt[gi][k]
        if cmin is None or c < cmin: mejor, cmin = s, c
    asig[gi] = mejor
    ac_i[mejor] += g_img[gi]
    for k in range(K): ac_c[mejor][k] += g_cnt[gi][k]

destino = {}
for gi, g in enumerate(grupos):
    for i in g: destino[i] = asig[gi]

for s in RATIO:
    for sub in ['images','labels']: (OUT/s/sub).mkdir(parents=True, exist_ok=True)
for i,(img,lbl,_) in enumerate(items):
    s = destino[i]
    shutil.copy2(img, OUT/s/'images'/img.name)
    shutil.copy2(lbl, OUT/s/'labels'/lbl.name)

yaml.safe_dump({'train':str(OUT/'train'/'images'), 'val':str(OUT/'valid'/'images'),
                'test':str(OUT/'test'/'images'), 'nc':K, 'names':names},
               open(OUT/'data.yaml','w'), sort_keys=False, allow_unicode=True)
DATA_YAML = str(OUT/'data.yaml')

# ---------- VERIFICACIÓN ----------
print(f"{'Split':<8}{'Imgs':>7}{'%':>7}   " + "".join(f"{nm.replace('billete_','$'):>8}" for nm in names))
for s in ['train','valid','test']:
    print(f"{s:<8}{ac_i[s]:>7}{100*ac_i[s]/n:>6.1f}%   " + "".join(f"{ac_c[s][k]:>8}" for k in range(K)))
print(f"{'ideal 15%':<8}{n*0.15:>7.0f}{'':>7}   " + "".join(f"{t*0.15:>8.0f}" for t in tot_c))

print("\nArchivos en disco:")
for s in ['train','valid','test']:
    print(f"  {s}: {len(list((OUT/s/'images').glob('*')))} imágenes")

flacas = {names[k] for k in range(K) for s in ['valid','test'] if ac_c[s][k] < 20}
print("\n⚠️  Clases con <20 instancias en valid o test:", flacas if flacas else "ninguna ✅")
print(f"Grupos: {len(grupos)} | Ninguna escena cruza de conjunto.")
print('data.yaml ->', DATA_YAML)

Split      Imgs      %       $100   $1000     $20    $200     $50    $500
train      1797  69.1%        413     284     368     285     308     511
valid       401  15.4%         83      60      73      61      65     103
test        402  15.5%         83      60      73      61      65     103
ideal 15%    390                87      61      77      61      66     108

Archivos en disco:
  train: 1797 imágenes
  valid: 401 imágenes
  test: 402 imágenes

⚠️  Clases con <20 instancias en valid o test: ninguna ✅
Grupos: 2416 | Ninguna escena cruza de conjunto.
data.yaml -> /content/dataset_scene/data.yaml


## 7. Fase 1


In [18]:
COMMON = dict(
    data=DATA_YAML, imgsz=640, batch=16, optimizer='AdamW',
    weight_decay=0.01, patience=10, seed=SEED, device=0, exist_ok=True,
    # augmentation: SOLO las del reporte
    degrees=15.0, fliplr=0.5, hsv_v=0.2, scale=0.1, mixup=0.2,
    # defaults de YOLOv8 que NO están en el reporte -> desactivados
    mosaic=0.0, hsv_h=0.0, hsv_s=0.0, translate=0.0,
    shear=0.0, perspective=0.0, flipud=0.0, copy_paste=0.0,
)
model = YOLO('yolov8s.pt')
model.train(**COMMON, epochs=10, lr0=1e-3, freeze=22,
            project='billetes', name='fase1_head')

Ultralytics 8.4.178 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_scene/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=22, hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=fase1_head, nbs=64, nms=None, opset=None, optimize=False, o

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7982c5a19f40>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

## 8.Fase 2



In [19]:
RUNS = '/content/runs/detect/billetes'

model = YOLO(f'{RUNS}/fase1_head/weights/best.pt')
model.train(**COMMON, epochs=15, lr0=1e-4, freeze=10,
            project='billetes', name='fase2_neck')

Ultralytics 8.4.178 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_scene/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=/content/runs/detect/billetes/fase1_head/weights/best.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=fase2_neck, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7983692f5860>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

## 9. Fase 3



In [20]:
model = YOLO(f'{RUNS}/fase2_neck/weights/best.pt')
model.train(**COMMON, epochs=25, lr0=1e-5, freeze=0,
            project='billetes', name='fase3_full')

Ultralytics 8.4.178 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_scene/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=1e-05, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=/content/runs/detect/billetes/fase2_neck/weights/best.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=fase3_full, nb

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7982946243d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

## 10. Evaluación sobre TEST



In [21]:
best = YOLO(f'{RUNS}/fase3_full/weights/best.pt')
m = best.val(data=DATA_YAML, split='test', plots=True)

print(f"mAP@0.5     : {m.box.map50:.4f}")
print(f"mAP@0.5:0.95: {m.box.map:.4f}")
print(f"Precision   : {m.box.mp:.4f}")
print(f"Recall      : {m.box.mr:.4f}\n")
print(f"{'Clase':<16}{'Precision':>11}{'Recall':>10}{'mAP@0.5':>10}")
for idx, c in enumerate(m.box.ap_class_index):
    print(f"{names[c]:<16}{m.box.p[idx]:>11.4f}{m.box.r[idx]:>10.4f}{m.box.ap50[idx]:>10.4f}")

Ultralytics 8.4.178 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 5.8±1.8 MB/s, size: 62.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/dataset_scene/test/labels... 402 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 402/402 441.8it/s 0.9s
val: New cache created: /content/dataset_scene/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 3.2it/s 8.0s
                   all        402        445       0.86      0.836      0.902      0.787
           billete_100         81         83      0.922      0.849      0.945      0.849
          billete_1000         58         60       0.76      0.685      0.775      0.675
          

## 11. Conteo del monto



In [22]:
VALOR = {'billete_20':20,'billete_50':50,'billete_100':100,
         'billete_200':200,'billete_500':500,'billete_1000':1000}

def contar(ruta):
    r = best(ruta, verbose=False)[0]
    c = Counter(r.names[int(x)] for x in r.boxes.cls.tolist())
    return dict(c), sum(VALOR[k]*v for k,v in c.items())

ejemplo = sorted((OUT/'test'/'images').glob('*'))[0]
det, total = contar(ejemplo)
print(ejemplo.name, '->', det, '| Monto: $', total)

IMG_2465_jpeg.rf.30f22635acfc128806e16d0df832c704.jpg -> {'billete_50': 1} | Monto: $ 50


## 12. Respaldar resultados



In [23]:
from google.colab import drive
drive.mount('/content/drive')
dst = '/content/drive/MyDrive/TT2_resultados'
shutil.copytree(f'{RUNS}/fase3_full', dst+'/fase3_full', dirs_exist_ok=True)
shutil.copy2(DATA_YAML, dst+'/data.yaml')
print('Respaldado en', dst)

Mounted at /content/drive
Respaldado en /content/drive/MyDrive/TT2_resultados


In [24]:
import random
random.seed(7)
tests = sorted((OUT/'test'/'images').glob('*'))
cands = []
for p in tests:
    lbl = OUT/'test'/'labels'/(p.stem + '.txt')
    nb = len([l for l in lbl.read_text().splitlines() if l.strip()])
    if nb >= 3: cands.append((nb, p))
cands.sort(reverse=True)
for nb, p in cands[:5]:
    det, total = contar(p)
    print(f"{p.name[:28]:<30} real={nb} billetes | detectado={det} | Monto: ${total}")

IMG_7907_JPG.rf.a63b08f037a5   real=5 billetes | detectado={'billete_20': 1, 'billete_100': 1, 'billete_50': 1, 'billete_500': 2} | Monto: $1170
IMG_5483_jpeg.rf.4ff99ffe394   real=5 billetes | detectado={'billete_500': 2, 'billete_200': 1} | Monto: $1200
IMG_5450_jpeg.rf.3d4012eda51   real=5 billetes | detectado={'billete_20': 1, 'billete_500': 2, 'billete_50': 1} | Monto: $1070
IMG_8133_JPG.rf.82822122e378   real=4 billetes | detectado={'billete_50': 3, 'billete_20': 1, 'billete_200': 1} | Monto: $370
IMG_7926_JPG.rf.000e8b8baa74   real=4 billetes | detectado={'billete_100': 2, 'billete_500': 2} | Monto: $1200


In [25]:
m25 = best.val(data=DATA_YAML, split='test', conf=0.25, plots=True,
               project='billetes', name='val_conf025')
print(f"mAP@0.5 (conf=0.25): {m25.box.map50:.4f}")

Ultralytics 8.4.178 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 994.0±220.2 MB/s, size: 49.6 KB)
val: Scanning /content/dataset_scene/test/labels.cache... 402 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 402/402 84.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 3.4it/s 7.7s
                   all        402        445       0.86      0.836      0.843      0.738
           billete_100         81         83      0.922      0.849      0.905      0.813
          billete_1000         58         60       0.76      0.685      0.683      0.594
            billete_20         72         73      0.913      0.863      0.869      0.751
           billete_200         60         61      0.839      0.885      0.884      0.775
            billete_50         63         

In [26]:
VALOR = {'billete_20':20,'billete_50':50,'billete_100':100,
         'billete_200':200,'billete_500':500,'billete_1000':1000}

tot = exactas_cnt = exactas_monto = 0
errs = []
for p in sorted((OUT/'test'/'images').glob('*')):
    lbl = OUT/'test'/'labels'/(p.stem+'.txt')
    filas = [l.split() for l in lbl.read_text().splitlines() if l.strip()]
    if len(filas) < 2: continue                      # solo escenas múltiples
    real_c = Counter(names[int(f[0])] for f in filas)
    real_m = sum(VALOR[k]*v for k,v in real_c.items())
    r = best(p, verbose=False)[0]
    det_c = Counter(r.names[int(c)] for c in r.boxes.cls.tolist())
    det_m = sum(VALOR[k]*v for k,v in det_c.items())
    tot += 1
    if sum(det_c.values()) == sum(real_c.values()): exactas_cnt += 1
    if det_m == real_m: exactas_monto += 1
    errs.append(abs(det_m - real_m))

print(f"Escenas múltiples evaluadas : {tot}")
print(f"Nº de billetes exacto       : {exactas_cnt}/{tot} = {100*exactas_cnt/tot:.1f}%")
print(f"Monto total exacto          : {exactas_monto}/{tot} = {100*exactas_monto/tot:.1f}%")
print(f"Error absoluto medio del monto: ${sum(errs)/len(errs):.0f}")

Escenas múltiples evaluadas : 22
Nº de billetes exacto       : 14/22 = 63.6%
Monto total exacto          : 8/22 = 36.4%
Error absoluto medio del monto: $230


In [27]:
from collections import Counter
dist = Counter()
for s in ['train','valid','test']:
    for lbl in (OUT/s/'labels').glob('*.txt'):
        nb = len([l for l in lbl.read_text().splitlines() if l.strip()])
        dist[nb] += 1
tot = sum(dist.values())
multi = sum(v for k,v in dist.items() if k >= 2)
print(f"{'Billetes/imagen':<18}{'Imágenes':>10}{'%':>8}")
for k in sorted(dist): print(f"{k:<18}{dist[k]:>10}{100*dist[k]/tot:>7.1f}%")
print(f"\nTotal imágenes        : {tot}")
print(f"Con 2+ billetes       : {multi}  ({100*multi/tot:.1f}%)")

Billetes/imagen     Imágenes       %
1                       2370   91.2%
2                         85    3.3%
3                         84    3.2%
4                         38    1.5%
5                         23    0.9%

Total imágenes        : 2600
Con 2+ billetes       : 230  (8.8%)


In [28]:
from collections import Counter
from pathlib import Path
import yaml
N = yaml.safe_load(open('/content/dataset_scene/data.yaml'))['names']
c = Counter()
for l in Path('/content/dataset_scene').rglob('labels/*.txt'):
    for ln in l.read_text().splitlines():
        if ln.strip(): c[N[int(ln.split()[0])]] += 1
for k in N: print(f'{k:14} {c[k]:5}')
print('TOTAL', sum(c.values()))

billete_100      579
billete_1000     404
billete_20       514
billete_200      407
billete_50       438
billete_500      717
TOTAL 3059


In [29]:
from google.colab import files
import shutil

# Matriz de confusión (la de conf=0.25, diagonal sana)
shutil.copy('/content/runs/detect/billetes/val_conf025/confusion_matrix_normalized.png',
            'matriz_confusion.png')
files.download('matriz_confusion.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
shutil.copy('/content/runs/detect/billetes/fase3_full/results.png',
            'curvas_entrenamiento.png')
files.download('curvas_entrenamiento.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
from ultralytics import YOLO
import cv2, matplotlib.pyplot as plt
from pathlib import Path

VALOR = {'billete_20':20,'billete_50':50,'billete_100':100,
         'billete_200':200,'billete_500':500,'billete_1000':1000}

m = YOLO('/content/runs/detect/billetes/fase3/weights/best.pt')

# <-- cambia esto por la imagen que elijas del test set
img = '/content/dataset_scene/test/images/TU_IMAGEN.jpg'

r = m.predict(img, conf=0.25, verbose=False)[0]
total = sum(VALOR[m.names[int(c)]] for c in r.boxes.cls)
n = len(r.boxes)

vis = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(12,9)); plt.imshow(vis); plt.axis('off')
plt.title(f'{n} billetes detectados  —  Total: ${total:,}', fontsize=15)
plt.savefig('ejemplo_conteo.png', dpi=150, bbox_inches='tight')
print(f'{n} billetes | Total ${total:,}')

FileNotFoundError: [Errno 2] No such file or directory: '/content/runs/detect/billetes/fase3/weights/best.pt'